# model-2: DenseNet121

Pasangan: Raja (trial 1-5) dan Faruq (trial 6-10). Ide kunci: **dense connection**.
Dibuat dari template notebook Kaggle.
Versi 2 (8 November 2026): data dibaca dari dataset Kaggle `simplicity-224-letterbox`.

**Pengaturan Kaggle (wajib):**
- *Settings > Accelerator*: GPU (T4 atau P100)
- *Settings > Internet*: **ON**, untuk `git clone` dan unduh bobot ImageNet (butuh verifikasi nomor HP)
- *Add Input* (panel kanan): cari dataset `simplicity-224-letterbox` milik `yonobengkel12`, lalu tambahkan. Isinya gambar 224×224 yang sudah jadi dan identik dengan hasil `data_utils.py`, jadi semua pasangan memakai input yang persis sama.
- Untuk pelatihan panjang, pakai *Save Version > Save & Run All* agar tetap berjalan walau browser ditutup.

**Aturan:** sel bertanda `JANGAN DIUBAH` dipakai apa adanya. `common/data_utils.py` dikunci dan hanya dipakai lewat import.

**Cara menjalankan (tiap orang di akun Kaggle masing-masing):**
1. Di sel *Pengaturan pasangan* (bagian 4), isi `NAMA` dan `TRIAL_SAYA` (Raja: 1-5, Faruq: 6-10). Sel lain tidak perlu diubah.
2. *Save Version > Save & Run All*. Kelima trial berjalan berurutan.
3. Dari tab *Output*, unduh `trials_<nama>.csv` dan `history_<nama>.json` (ada di `simplicity-cnn-comparison/model-2_densenet121/`), lalu commit ke folder `model-2_densenet121/`.
   Berkas `model-2_trialXX.keras` (model terbaik milikmu) **tidak** di-commit; biarkan di Output Kaggle.
4. Setelah kedua orang selesai, sel *Gabung hasil* menulis `trials.csv` dan `history_densenet121.png`.

## 1. Setup bersama (JANGAN DIUBAH)
Clone repo (di Kaggle) atau cari root repo (di lokal), import `data_utils`, cetak sidik versinya, lalu cek perangkat.
Sidik yang tercetak harus sama dengan sidik resmi di README bagian 3 (per 8 November 2026: `b3e0629783c5`).

In [ ]:
# ===== SETUP BERSAMA: JANGAN DIUBAH =====
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/0Raja0-RA/simplicity-cnn-comparison.git"
BRANCH = "main"

if Path("/kaggle/working").exists():                      # di Kaggle
    ROOT = Path("/kaggle/working/simplicity-cnn-comparison")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, str(ROOT)], check=True)
else:                                                     # di lokal: naik sampai ketemu root repo
    ROOT = Path.cwd().resolve()
    while not (ROOT / "common" / "data_utils.py").exists():
        if ROOT.parent == ROOT:
            raise FileNotFoundError("Root repo (berisi common/data_utils.py) tidak ditemukan.")
        ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
from common import data_utils as du

SIDIK = du.sidik_config()
INFO_PERANGKAT = du.cek_perangkat()
print("ROOT:", ROOT)

## 2. Data (JANGAN DIUBAH, kecuali `BATCH_SIZE`)
Gambar float32 0-255 berukuran `du.IMG_SIZE`, sudah di-padding, belum dinormalisasi. Label integer 0-9
(pakai `sparse_categorical_crossentropy`). Augmentasi **tidak** ada di dataset; pasang `du.get_augmentation()` di model.
`du.load_data()` mencetak sumber data. Di Kaggle harus tertulis `simplicity_224.npz (npz dataset Kaggle)`; sel ini berhenti dengan pesan galat kalau dataset belum dipasang.

In [ ]:
BATCH_SIZE = 32          # boleh diubah per pasangan (README bagian 5)

# Di Kaggle, dataset simplicity-224-letterbox wajib terpasang (Add Input) supaya semua pasangan memakai gambar yang sama persis
if Path("/kaggle/working").exists() and not any(Path("/kaggle/input").rglob(du.NPZ_NAMA)):
    raise RuntimeError(f"Dataset Kaggle {du.KAGGLE_DATASET} belum dipasang. "
                       "Tambahkan lewat Add Input di panel kanan, lalu jalankan ulang sel ini.")

train_ds, val_ds = du.load_data(batch_size=BATCH_SIZE)   # mencetak sumber data: harus simplicity_224.npz
CLASS_NAMES = du.get_class_names()
print("IMG_SIZE:", du.IMG_SIZE, "| kelas:", CLASS_NAMES)

## 3. Model (diisi pasangan): DenseNet121
**Ide kunci: dense connection.** Di dalam satu dense block, setiap layer menerima gabungan (*concatenate*) feature map
dari **semua** layer sebelumnya, bukan hanya layer tepat sebelumnya (berbeda dengan ResNet yang *menjumlahkan*).
DenseNet121 punya 4 dense block (6, 12, 24, 16 layer, growth rate 32) yang dipisah *transition layer*
(BN, konvolusi 1x1, average pooling) untuk memadatkan channel. Akibatnya: fitur dipakai ulang (*feature reuse*),
gradien mengalir lancar, dan parameternya sedikit (sekitar 7 juta tanpa head).

Di Keras, nama layer mengikuti blok: `conv2_*` = dense block 1, `conv3_*` = block 2, `conv4_*` = block 3,
`conv5_*` = block 4, dan `pool2_`/`pool3_`/`pool4_` = transition layer.

Urutan di dalam model: `Input` -> `du.get_augmentation()` -> `densenet.preprocess_input` -> DenseNet121 ImageNet
(`include_top=False`, `pooling="avg"`) -> Dropout -> (opsional Dense ReLU -> Dropout) -> Dense 10 softmax.

In [ ]:
from tensorflow import keras

# Layer DenseNet121 yang dibuka (trainable) per strategi; transition layer ikut blok sesudahnya
AWALAN_DIBUKA = {
    "beku": (),                                                # hanya head yang dilatih
    "block4": ("pool4_", "conv5_"),                            # dense block 4
    "block3+4": ("pool3_", "conv4_", "pool4_", "conv5_"),      # dense block 3 dan 4
}


def bangun_model(cfg):
    """DenseNet121 + head 10 kelas sesuai satu konfigurasi trial, sudah di-compile."""
    du.set_seed()                                   # bobot head awal sama di akun mana pun
    inputs = keras.Input(shape=du.IMG_SIZE + (3,))
    x = du.get_augmentation()(inputs)               # wajib, aktif hanya saat training

    # DenseNet di Keras TIDAK menormalisasi di dalam dirinya, jadi preprocess_input wajib dipasang.
    # Mode "torch": piksel / 255, lalu dikurangi mean dan dibagi std ImageNet per channel.
    x = keras.applications.densenet.preprocess_input(x)

    base = keras.applications.DenseNet121(include_top=False, weights="imagenet",
                                          input_shape=du.IMG_SIZE + (3,), pooling="avg")
    awalan = AWALAN_DIBUKA[cfg["buka_layer"]]
    base.trainable = bool(awalan)
    for layer in base.layers:
        # BatchNormalization tetap beku: batch kecil (16/32) akan merusak statistik ImageNet
        layer.trainable = (bool(awalan) and layer.name.startswith(awalan)
                           and not isinstance(layer, keras.layers.BatchNormalization))
    x = base(x, training=False)                     # BN selalu memakai statistik ImageNet

    x = keras.layers.Dropout(cfg["dropout"])(x)
    if cfg["unit_head"]:
        x = keras.layers.Dense(cfg["unit_head"], activation="relu")(x)
        x = keras.layers.Dropout(cfg["dropout"])(x)
    outputs = keras.layers.Dense(len(CLASS_NAMES), activation="softmax")(x)
    model = keras.Model(inputs, outputs, name="densenet121_simplicity")

    if cfg["optimizer"] == "adamw":
        opt = keras.optimizers.AdamW(learning_rate=cfg["learning_rate"], weight_decay=cfg["weight_decay"])
    else:
        opt = keras.optimizers.Adam(learning_rate=cfg["learning_rate"])
    model.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

## 4. Tuning: random search (diisi pasangan)
Ikuti README bagian 5: **10 trial**, maksimal **40 epoch**, `EarlyStopping(val_loss, patience=8, restore_best_weights=True)`.
Pilih konfigurasi terbaik dari **validation** saja (val_acc tertinggi; kalau seri, val_loss terendah).

**Pembagian trial:** ke-10 konfigurasi diundi sekaligus di awal dengan seed 42, jadi daftarnya sama persis di akun
Raja dan Faruq (cocokkan *sidik konfigurasi* yang tercetak). Raja menjalankan trial 1-5, Faruq trial 6-10.
Hasil tiap orang dicatat di `trials_<nama>.csv`. Trial yang sudah tercatat dilewati, jadi kalau sesi terputus,
cukup commit/unggah `trials_<nama>.csv` dan `history_<nama>.json` yang sudah ada lalu jalankan ulang.

In [ ]:
# ===== PENGATURAN PASANGAN: hanya dua baris ini yang diubah =====
NAMA = "raja"                    # "raja" atau "faruq"
TRIAL_SAYA = [1, 2, 3, 4, 5]     # Raja: [1, 2, 3, 4, 5]   Faruq: [6, 7, 8, 9, 10]
# ================================================================

import csv
import hashlib
import json
import shutil
import time

import numpy as np
import pandas as pd

ARSITEKTUR = "densenet121"
N_TRIAL = 10                     # README bagian 5
MAX_EPOCH = 40
PATIENCE = 8
assert set(TRIAL_SAYA) <= set(range(1, N_TRIAL + 1)), "TRIAL_SAYA harus di antara 1-10"

MODEL_DIR = ROOT / "model-2_densenet121"
MODEL_DIR.mkdir(exist_ok=True)
TRIALS_SAYA_CSV = MODEL_DIR / f"trials_{NAMA}.csv"
HISTORY_SAYA_JSON = MODEL_DIR / f"history_{NAMA}.json"

# Berkas .keras disimpan DI LUAR folder repo (aturan 9): /kaggle/working di Kaggle
SIMPAN_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else ROOT.parent / "model_tersimpan_simplicity"
SIMPAN_DIR.mkdir(exist_ok=True)

# Melanjutkan sesi: kalau trials_<nama>.csv / history_<nama>.json diunggah lewat Add Input, salin ke model-2_densenet121/
if Path("/kaggle/input").exists():
    for berkas in (TRIALS_SAYA_CSV, HISTORY_SAYA_JSON):
        if not berkas.exists():
            temuan = sorted(Path("/kaggle/input").rglob(berkas.name))
            if temuan:
                shutil.copy(temuan[0], berkas)
                print("Dilanjutkan dari:", temuan[0])

print(f"{NAMA} menjalankan trial {TRIAL_SAYA}")
print("Log trial :", TRIALS_SAYA_CSV)
print("Model     :", SIMPAN_DIR)

In [ ]:
# Ruang pencarian DenseNet121. Diundi sekali untuk ke-10 trial dengan seed 42.
#  - buka_layer : membuka per dense block lebih bermakna daripada persen layer
#  - learning_rate (log-uniform): head saja boleh besar; kalau backbone dibuka harus kecil
#    supaya bobot ImageNet tidak rusak
#  - weight_decay hanya berlaku untuk AdamW (Adam = 0)
#  - dropout 0.2-0.5 dan head kecil: data train hanya 500 gambar
#  - batch_size 16/32: aman untuk memori GPU T4/P100 di 224x224
def sampel_konfigurasi(rng):
    buka = str(rng.choice(["beku", "block4", "block3+4"]))
    if buka == "beku":
        lr = 10 ** rng.uniform(np.log10(1e-4), np.log10(3e-3))
    else:
        lr = 10 ** rng.uniform(np.log10(2e-5), np.log10(2e-4))
    optimizer = str(rng.choice(["adam", "adamw"]))
    weight_decay = float(rng.choice([1e-5, 1e-4, 1e-3])) if optimizer == "adamw" else 0.0
    return {
        "buka_layer": buka,
        "learning_rate": float(f"{lr:.2e}"),
        "optimizer": optimizer,
        "weight_decay": weight_decay,
        "dropout": round(float(rng.uniform(0.2, 0.5)), 2),
        "unit_head": int(rng.choice([0, 256])),
        "batch_size": int(rng.choice([16, 32])),
    }


_rng = np.random.default_rng(du.SEED)
KONFIGURASI = {t: sampel_konfigurasi(_rng) for t in range(1, N_TRIAL + 1)}
PARAMETER = list(KONFIGURASI[1])

SIDIK_KONFIG = hashlib.sha256(json.dumps(KONFIGURASI, sort_keys=True).encode()).hexdigest()[:12]
print("Sidik konfigurasi:", SIDIK_KONFIG, "(harus sama di akun Raja dan Faruq)")
pd.DataFrame.from_dict(KONFIGURASI, orient="index").rename_axis("trial")

In [ ]:
KOLOM = ["trial", *PARAMETER, "val_acc", "val_loss", "best_epoch", "epoch_dijalankan",
         "train_time_sec", "dijalankan_oleh", "sidik_config"]


def baca_trials(path):
    if not path.exists():
        return []
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def skor(row):
    """Urutan pemilihan: val_acc tertinggi, kalau seri val_loss terendah."""
    return (float(row["val_acc"]), -float(row["val_loss"]))


riwayat_saya = json.loads(HISTORY_SAYA_JSON.read_text(encoding="utf-8")) if HISTORY_SAYA_JSON.exists() else {}
log_saya = baca_trials(TRIALS_SAYA_CSV)
selesai = {int(r["trial"]) for r in log_saya}
terbaik_saya = max(log_saya, key=skor) if log_saya else None

for t in TRIAL_SAYA:
    if t in selesai:
        print(f"Trial {t}: sudah tercatat, dilewati")
        continue
    cfg = KONFIGURASI[t]
    print(f"\n===== Trial {t}/{N_TRIAL}: {cfg}")

    keras.backend.clear_session()
    train_ds, val_ds = du.load_data(batch_size=cfg["batch_size"])   # juga memanggil set_seed(42)
    model = bangun_model(cfg)
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                       restore_best_weights=True, verbose=1)
    mulai = time.time()
    hist = model.fit(train_ds, validation_data=val_ds, epochs=MAX_EPOCH, callbacks=[es], verbose=2)
    durasi = time.time() - mulai

    val_loss, val_acc = model.evaluate(val_ds, verbose=0)           # bobot terbaik (sudah di-restore)
    row = {"trial": t, **cfg,
           "val_acc": round(float(val_acc), 4), "val_loss": round(float(val_loss), 4),
           "best_epoch": int(np.argmin(hist.history["val_loss"])) + 1,
           "epoch_dijalankan": len(hist.history["val_loss"]),
           "train_time_sec": round(durasi, 1), "dijalankan_oleh": NAMA, "sidik_config": SIDIK[:12]}

    baru = not TRIALS_SAYA_CSV.exists()
    with open(TRIALS_SAYA_CSV, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=KOLOM, lineterminator="\n")
        if baru:
            w.writeheader()
        w.writerow(row)
    riwayat_saya[str(t)] = {k: [round(float(v), 5) for v in vals] for k, vals in hist.history.items()}
    HISTORY_SAYA_JSON.write_text(json.dumps(riwayat_saya, indent=1), encoding="utf-8")
    print(f"Trial {t}: val_acc {row['val_acc']:.4f}, val_loss {row['val_loss']:.4f}, "
          f"epoch terbaik {row['best_epoch']}, {durasi / 60:.1f} menit")

    # Simpan hanya model terbaik milik sendiri (di luar repo), untuk evaluasi test nanti
    row_str = {k: str(v) for k, v in row.items()}
    if terbaik_saya is None or skor(row_str) > skor(terbaik_saya):
        if terbaik_saya is not None:
            (SIMPAN_DIR / f"model-2_trial{int(terbaik_saya['trial']):02d}.keras").unlink(missing_ok=True)
        model.save(SIMPAN_DIR / f"model-2_trial{t:02d}.keras")
        terbaik_saya = row_str
        print(f"Model trial {t} disimpan sebagai model terbaik milik {NAMA}")

pd.read_csv(TRIALS_SAYA_CSV)

### Gabung hasil pasangan
Membaca semua `trials_*.csv` di `model-2_densenet121/`. Kalau kesepuluh trial sudah lengkap, sel ini menulis `trials.csv`
dan `history_densenet121.png` (grafik loss dan akurasi train vs validation dari trial terbaik).
Sel ini tidak butuh GPU, jadi boleh dijalankan di lokal setelah `git pull`.

In [ ]:
import matplotlib.pyplot as plt

semua = {}
for p in sorted(MODEL_DIR.glob("trials_*.csv")):
    for r in baca_trials(p):
        semua[int(r["trial"])] = r
riwayat = {}
for p in sorted(MODEL_DIR.glob("history_*.json")):
    if p.name != f"history_{ARSITEKTUR}.json":
        riwayat.update(json.loads(p.read_text(encoding="utf-8")))

kurang = sorted(set(range(1, N_TRIAL + 1)) - set(semua))
lengkap = not kurang
print("Trial tercatat:", sorted(semua), "| belum:", kurang if kurang else "-")

if semua:
    df = pd.DataFrame([semua[t] for t in sorted(semua)])
    terbaik = max(semua.values(), key=skor)
    t_best = int(terbaik["trial"])
    print(f"\nTrial terbaik{'' if lengkap else ' (SEMENTARA)'}: {t_best} oleh {terbaik['dijalankan_oleh']}, "
          f"val_acc {float(terbaik['val_acc']):.4f}, val_loss {float(terbaik['val_loss']):.4f}")
    print("Konfigurasi:", {k: terbaik[k] for k in PARAMETER})

    h = riwayat.get(str(t_best))
    if h:
        ep = range(1, len(h["loss"]) + 1)
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        for i, (m, judul) in enumerate([("loss", "Loss"), ("accuracy", "Akurasi")]):
            ax[i].plot(ep, h[m], label="train")
            ax[i].plot(ep, h[f"val_{m}"], label="validation")
            ax[i].axvline(int(terbaik["best_epoch"]), color="gray", ls="--", lw=1, label="epoch terbaik")
            ax[i].set(title=f"{judul} DenseNet121 (trial {t_best})", xlabel="epoch", ylabel=judul.lower())
            ax[i].legend()
        fig.tight_layout()
        if lengkap:
            fig.savefig(MODEL_DIR / f"history_{ARSITEKTUR}.png", dpi=120)
        plt.show()

    if lengkap:
        df.to_csv(MODEL_DIR / "trials.csv", index=False, lineterminator="\n")
        print("Ditulis:", MODEL_DIR / "trials.csv", "dan", MODEL_DIR / f"history_{ARSITEKTUR}.png")
        print("Langkah berikutnya: tulis konfigurasi terbaik di model-2_densenet121/README.md, commit, baru evaluasi test.")
    display(df)

## 5. Evaluasi test (SEL PALING AKHIR)
Jalankan **satu kali** saja, setelah konfigurasi terbaik ditulis dan di-commit di `model-N/README.md`.
Ubah `JALANKAN_TEST = True` hanya pada saat itu. Hasil test tidak boleh dipakai untuk mengubah apa pun.
Dijalankan oleh **pemilik trial terbaik** (lihat sel *Gabung hasil*), karena model terbaiknya ada di Output Kaggle
miliknya (`model-2_trialXX.keras`). Kalau sesinya sudah berganti, unggah berkas itu lewat *Add Input* lebih dulu.

In [ ]:
JALANKAN_TEST = False

if JALANKAN_TEST:
    assert (MODEL_DIR / "trials.csv").exists(), "trials.csv belum ada: tuning belum lengkap"
    terbaik = max(baca_trials(MODEL_DIR / "trials.csv"), key=skor)
    t_best = int(terbaik["trial"])
    nama_model = f"model-2_trial{t_best:02d}.keras"
    kandidat = [SIMPAN_DIR / nama_model]
    if Path("/kaggle/input").exists():
        kandidat += sorted(Path("/kaggle/input").rglob(nama_model))
    ada = [p for p in kandidat if p.exists()]
    if not ada:
        raise FileNotFoundError(f"{nama_model} tidak ditemukan. Jalankan sel ini di akun {terbaik['dijalankan_oleh']}, "
                                "atau unggah berkas itu dari Output Kaggle lewat Add Input.")
    path_model = ada[0]
    model_terbaik = keras.models.load_model(path_model)
    print("Model:", path_model, "| trial", t_best, {k: terbaik[k] for k in PARAMETER})

    test_ds = du.load_test()
    y_true = du.ambil_label(test_ds)
    model_terbaik.predict(test_ds.take(1), verbose=0)                 # pemanasan GPU, tidak diukur
    mulai = time.time()
    y_pred = model_terbaik.predict(test_ds, verbose=0).argmax(1)
    inferensi_ms = (time.time() - mulai) / len(y_true) * 1000

    cm = np.zeros((len(CLASS_NAMES),) * 2, dtype=int)
    np.add.at(cm, (y_true, y_pred), 1)                                 # baris = asli, kolom = prediksi
    tp = np.diag(cm)
    presisi = np.divide(tp, cm.sum(0), out=np.zeros(len(tp)), where=cm.sum(0) > 0)
    recall = np.divide(tp, cm.sum(1), out=np.zeros(len(tp)), where=cm.sum(1) > 0)
    f1 = np.divide(2 * presisi * recall, presisi + recall, out=np.zeros(len(tp)), where=(presisi + recall) > 0)

    hasil = {
        "arsitektur": "DenseNet121", "trial_terbaik": t_best,
        "test_acc": round(float((y_true == y_pred).mean()), 4),
        "macro_f1": round(float(f1.mean()), 4),
        "jumlah_parameter": int(model_terbaik.count_params()),
        "parameter_dilatih": int(sum(np.prod(w.shape) for w in model_terbaik.trainable_weights)),
        "ukuran_model_mb": round(path_model.stat().st_size / 2**20, 1),
        "inferensi_ms_per_gambar": round(inferensi_ms, 2),
        "train_time_sec": float(terbaik["train_time_sec"]),
        "sidik_config": SIDIK[:12],
    }
    (MODEL_DIR / "hasil_test.json").write_text(json.dumps(hasil, indent=2), encoding="utf-8")
    print(json.dumps(hasil, indent=2))
    display(pd.DataFrame({"presisi": presisi, "recall": recall, "f1": f1}, index=CLASS_NAMES).round(3))

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set(xlabel="prediksi", ylabel="asli", title=f"Confusion matrix DenseNet121 (test, akurasi {hasil['test_acc']:.1%})")
    fig.tight_layout()
    fig.savefig(MODEL_DIR / f"confusion_{ARSITEKTUR}.png", dpi=120)
    plt.show()
else:
    print("Evaluasi test belum dijalankan (JALANKAN_TEST = False).")